<a href="https://colab.research.google.com/github/D2718281828nis/BioMedAI-sEEG-core-of-epilepsy/blob/main/sEEG_DWT_viewer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive DWT and CWT viewer for sEEG (Google Colab)

Explore a local **`sEEG-HFOs-8.edf`** with two complementary wavelet views:

- **DWT**: a multiresolution, energy-preserving decomposition with time-aligned reconstructed components.
- **CWT**: a time–frequency scalogram for transient activity across a configurable frequency range.

The viewer excludes channels beginning with `MKR`, reads only the selected EDF segment, and never filters or resamples the source recording. Run the cells from top to bottom, then use the tabs in the final cell.

> **Interpretation:** dyadic DWT band limits and CWT frequencies are analytical approximations, not clinical frequency measurements. Boundary regions are shaded because wavelet coefficients there are affected by padding. Confirm findings against the unprocessed trace and your acquisition/filter settings.


## 1. Install dependencies

The pinned lower bounds document the APIs used while allowing Colab's compatible preinstalled packages to remain in place.


In [1]:
%pip install -q "mne>=1.6" "edfio>=0.4" "PyWavelets>=1.5" "ipywidgets>=8.0" "scipy>=1.10"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## 2. Imports and display defaults


In [2]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import mne
import numpy as np
import pywt
from IPython.display import display
from scipy import signal as scipy_signal

mne.set_log_level("WARNING")
plt.rcParams.update({
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.figsize": (15, 8),
})


## 3. Locate or upload the EDF

The browser upload is used only in Colab. In another Jupyter environment, place the EDF in the working directory or its `dataset` subdirectory.


In [3]:
EDF_NAME = "sEEG-HFOs-8.edf"
candidates = [
    Path("/content") / EDF_NAME,
    Path("/content/dataset") / EDF_NAME,
    Path.cwd() / EDF_NAME,
    Path.cwd() / "dataset" / EDF_NAME,
]
edf_path = next((path for path in candidates if path.is_file()), None)

if edf_path is None:
    try:
        from google.colab import files
    except ImportError as exc:
        searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
        raise FileNotFoundError(f"{EDF_NAME} was not found. Searched:\n{searched}") from exc

    print(f"Select {EDF_NAME} in the upload dialog.")
    uploaded = files.upload()
    if EDF_NAME not in uploaded:
        raise FileNotFoundError(f"The uploaded file must be named {EDF_NAME!r}.")
    edf_path = Path("/content") / EDF_NAME

print(f"EDF file: {edf_path.resolve()}")


EDF file: /content/sEEG-HFOs-8.edf


## 4. Load metadata and select signal channels

`preload=False` keeps the recording disk-backed. Marker matching is deliberately case-insensitive and prefix-based to cover labels such as `MKR1`, `MKR 2`, and `MKR1+`. Verify the retained list before interpreting results.


In [4]:
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING")
marker_channels = [name for name in raw.ch_names if name.strip().upper().startswith("MKR")]
signal_channels = [name for name in raw.ch_names if name not in marker_channels]
if not signal_channels:
    raise ValueError("No non-MKR signal channels were found in the EDF.")

raw.pick(signal_channels)
sfreq = float(raw.info["sfreq"])
duration_s = raw.n_times / sfreq
nyquist_hz = sfreq / 2

print(f"Signal channels: {len(signal_channels)}")
print(f"Excluded markers: {marker_channels or 'none found'}")
print(f"Sampling frequency: {sfreq:g} Hz | Nyquist: {nyquist_hz:g} Hz")
print(f"Samples: {raw.n_times:,} | Duration: {duration_s:.2f} s")
display(signal_channels)


/tmp/ipykernel_2154/3227331436.py:1: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING")
/tmp/ipykernel_2154/3227331436.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING")


Signal channels: 100
Excluded markers: ['MKR1+', 'MKR2+']
Sampling frequency: 256 Hz | Nyquist: 128 Hz
Samples: 712,448 | Duration: 2783.00 s


['EEG R1',
 'EEG R2',
 'EEG R3',
 'EEG R4',
 'EEG R5',
 'EEG R6',
 'EEG R7',
 'EEG R8',
 'EEG R9',
 'EEG R10',
 'EEG FP1',
 'EEG FP2',
 'EEG FP3',
 'EEG FP4',
 'EEG FP5',
 'EEG FP6',
 'EEG FP7',
 'EEG FP8',
 'EEG FD1',
 'EEG FD2',
 'EEG FD3',
 'EEG FD4',
 'EEG FD5',
 'EEG FD6',
 'EEG PM1',
 'EEG PM2',
 'EEG PM3',
 'EEG PM4',
 'EEG PM5',
 'EEG PM6',
 'EEG PM7',
 'EEG PM8',
 'EEG CC1',
 'EEG CC2',
 'EEG CC3',
 'EEG CC4',
 'EEG CC5',
 'EEG CC6',
 'EEG CC7',
 'EEG CC8',
 'EEG CC9',
 'EEG CC10',
 'EEG SA1',
 'EEG SA2',
 'EEG SA3',
 'EEG SA4',
 'EEG SA5',
 'EEG SA6',
 'EEG PA1',
 'EEG PA2',
 'EEG PA3',
 'EEG PA4',
 'EEG PA5',
 'EEG PA6',
 'EEG PA7',
 'EEG PA8',
 'EEG PA9',
 'EEG PA10',
 "EEG CR'1",
 "EEG CR'2",
 "EEG CR'3",
 "EEG CR'4",
 "EEG CR'5",
 "EEG CR'6",
 "EEG CR'7",
 "EEG CR'8",
 "EEG CR'9",
 "EEG CR'10",
 "EEG CC'1",
 "EEG CC'2",
 "EEG CC'3",
 "EEG CC'4",
 "EEG CC'5",
 "EEG CC'6",
 "EEG CC'7",
 "EEG CC'8",
 "EEG CC'9",
 "EEG CC'10",
 "EEG PM'1",
 "EEG PM'2",
 "EEG PM'3",
 "EEG PM'4

## 5. Analysis helpers

Segments use integer sample bounds, avoiding the inclusive-`tmax` ambiguity of time-based slicing. Optional preprocessing is applied to a copy of the displayed segment only. DWT reconstruction uses the same extension mode for decomposition and synthesis; CWT scales are derived from requested physical frequencies and the actual sampling interval.


In [5]:
DWT_MODE = "symmetric"
CWT_VOICES_PER_OCTAVE = 12


def read_segment(channel, start_s, window_s, preprocessing):
    """Read exactly one bounded segment and return time and microvolt arrays."""
    start_sample = int(np.clip(round(start_s * sfreq), 0, raw.n_times - 1))
    sample_count = max(2, int(round(window_s * sfreq)))
    stop_sample = min(start_sample + sample_count, raw.n_times)
    data = raw.get_data(picks=[channel], start=start_sample, stop=stop_sample)[0] * 1e6
    times = np.arange(start_sample, stop_sample, dtype=float) / sfreq
    if preprocessing == "Remove mean":
        data = data - np.mean(data)
    elif preprocessing == "Linear detrend":
        data = scipy_signal.detrend(data, type="linear")
    return times, data


def reconstructed_dwt_components(values, wavelet_name, requested_level):
    """Return full-length approximation/detail reconstructions and the used level."""
    wavelet = pywt.Wavelet(wavelet_name)
    maximum_level = pywt.dwt_max_level(len(values), wavelet.dec_len)
    level = min(int(requested_level), maximum_level)
    if level < 1:
        raise ValueError(
            f"The segment is too short for {wavelet_name!r}; choose a longer window "
            "or a shorter-support wavelet."
        )
    coefficients = pywt.wavedec(values, wavelet, level=level, mode=DWT_MODE)
    components = []
    for index, coefficient in enumerate(coefficients):
        isolated = [np.zeros_like(item) for item in coefficients]
        isolated[index] = coefficient
        components.append(pywt.waverec(isolated, wavelet, mode=DWT_MODE)[: len(values)])
    labels = [f"A{level}"] + [f"D{detail}" for detail in range(level, 0, -1)]
    return labels, components, level, wavelet


def dwt_frequency_range(label):
    """Return the idealized dyadic passband in Hz."""
    level = int(label[1:])
    if label.startswith("A"):
        return 0.0, sfreq / 2 ** (level + 1)
    return sfreq / 2 ** (level + 1), sfreq / 2 ** level


def cwt_power(values, wavelet_name, minimum_hz, maximum_hz):
    """Compute log-spaced CWT power at approximately constant voices per octave."""
    if not 0 < minimum_hz < maximum_hz < nyquist_hz:
        raise ValueError(f"Require 0 < minimum < maximum < Nyquist ({nyquist_hz:g} Hz).")
    octaves = np.log2(maximum_hz / minimum_hz)
    count = max(16, int(np.ceil(octaves * CWT_VOICES_PER_OCTAVE)) + 1)
    frequencies = np.geomspace(maximum_hz, minimum_hz, count)
    central_frequency = pywt.central_frequency(wavelet_name)
    scales = central_frequency * sfreq / frequencies
    coefficients, actual_frequencies = pywt.cwt(
        values, scales, wavelet_name, sampling_period=1 / sfreq, method="fft"
    )
    return np.abs(coefficients) ** 2, actual_frequencies, scales


def shade_boundaries(axis, times, edge_seconds):
    """Mark approximate edge-affected regions without claiming a strict CWT COI."""
    edge_seconds = min(edge_seconds, (times[-1] - times[0]) / 2)
    if edge_seconds > 0:
        axis.axvspan(times[0], times[0] + edge_seconds, color="white", alpha=0.22, lw=0)
        axis.axvspan(times[-1] - edge_seconds, times[-1], color="white", alpha=0.22, lw=0)


## 6. Shared controls

Updates occur on slider release to avoid repeatedly transforming data while a control is dragged. Window changes constrain the start time automatically.


In [6]:
control_style = {"description_width": "initial"}
channel_picker = widgets.Dropdown(
    options=signal_channels, value=signal_channels[0], description="Channel:",
    layout=widgets.Layout(width="360px"),
)
window_picker = widgets.Dropdown(
    options=[1.0, 2.0, 5.0, 10.0, 20.0, 30.0], value=5.0,
    description="Window (s):", style=control_style,
)
start_picker = widgets.FloatSlider(
    value=0.0, min=0.0, max=max(0.0, duration_s - window_picker.value),
    step=max(1 / sfreq, min(0.25, duration_s / 4000)), description="Start (s):",
    readout_format=".3f", continuous_update=False,
    layout=widgets.Layout(width="98%"),
)
preprocessing_picker = widgets.ToggleButtons(
    options=["Raw segment", "Remove mean", "Linear detrend"], value="Remove mean",
    description="Preprocessing:", style=control_style,
)


def update_start_range(change=None):
    start_picker.max = max(0.0, duration_s - float(window_picker.value))
    start_picker.value = min(start_picker.value, start_picker.max)


window_picker.observe(update_start_range, names="value")
shared_controls = widgets.VBox([
    widgets.HBox([channel_picker, window_picker]),
    preprocessing_picker,
    start_picker,
])


## 7. DWT view

The original segment and all reconstructed components share the original time grid. The title reports when a requested level is reduced to the maximum valid level. Boundary shading is based on the wavelet filter support at each component level.


In [7]:
dwt_wavelet_picker = widgets.Dropdown(
    options=pywt.wavelist(kind="discrete"), value="db4", description="Mother wavelet:",
    style=control_style, layout=widgets.Layout(width="300px"),
)
dwt_level_picker = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="Requested levels:",
    continuous_update=False, style=control_style,
)


def plot_dwt(channel, start_s, window_s, preprocessing, wavelet_name, requested_level):
    times, values = read_segment(channel, start_s, window_s, preprocessing)
    try:
        labels, components, used_level, wavelet = reconstructed_dwt_components(
            values, wavelet_name, requested_level
        )
    except ValueError as error:
        print(f"DWT unavailable: {error}")
        return

    figure, axes = plt.subplots(
        len(components) + 1, 1,
        figsize=(15, max(8, 1.65 * (len(components) + 1))),
        sharex=True, constrained_layout=True,
    )
    axes[0].plot(times, values, color="black", linewidth=0.75)
    axes[0].set_ylabel("Signal\n(µV)")
    level_note = "" if used_level == requested_level else f"; reduced from {requested_level}"
    axes[0].set_title(
        f"{channel} — DWT ({wavelet_name}, level {used_level}{level_note}) | "
        f"{times[0]:.3f}–{times[-1] + 1/sfreq:.3f} s | {preprocessing}"
    )

    for axis, label, component in zip(axes[1:], labels, components):
        low, high = dwt_frequency_range(label)
        axis.plot(times, component, linewidth=0.7)
        axis.set_ylabel(f"{label}\n≈{low:g}–{high:g} Hz\n(µV)")
        support_samples = (wavelet.dec_len - 1) * (2 ** int(label[1:]))
        shade_boundaries(axis, times, support_samples / sfreq)

    axes[-1].set_xlabel("Time (s)")
    for axis in axes:
        axis.margins(x=0)
    plt.show()


dwt_output = widgets.interactive_output(plot_dwt, {
    "channel": channel_picker, "start_s": start_picker, "window_s": window_picker,
    "preprocessing": preprocessing_picker, "wavelet_name": dwt_wavelet_picker,
    "requested_level": dwt_level_picker,
})
dwt_tab = widgets.VBox([
    widgets.HBox([dwt_wavelet_picker, dwt_level_picker]),
    widgets.HTML("<small>Shaded ends are padding-sensitive. Band labels are idealized dyadic limits.</small>"),
    dwt_output,
])


## 8. CWT view

The scalogram uses logarithmically spaced frequencies (12 voices per octave), FFT convolution, log-normalized power, and robust 1st–99th percentile color limits. The trace remains visible above the scalogram. Lower frequencies have wider edge-affected regions; the shading uses the largest selected scale as a conservative visual warning, not a formal cone of influence.


In [9]:
continuous_wavelets = [name for name in pywt.wavelist(kind="continuous") if name != "gaus"]
cwt_wavelet_picker = widgets.Dropdown(
    options=continuous_wavelets, value="cmor", description="Mother wavelet:",
    style=control_style, layout=widgets.Layout(width="320px"),
)
frequency_max_limit = max(0.2, nyquist_hz * 0.95)
frequency_min_default = min(1.0, frequency_max_limit / 4)
frequency_max_default = min(150.0, nyquist_hz * 0.9)
if frequency_max_default <= frequency_min_default:
    frequency_min_default = frequency_max_default / 4

cwt_min_picker = widgets.FloatLogSlider(
    value=frequency_min_default, base=10, min=np.log10(max(0.05, 1 / duration_s)),
    max=np.log10(max(0.06, frequency_max_limit / 2)), step=0.05,
    description="Minimum (Hz):", readout_format=".2f", continuous_update=False,
    style=control_style, layout=widgets.Layout(width="380px"),
)
cwt_max_picker = widgets.FloatLogSlider(
    value=frequency_max_default, base=10, min=np.log10(max(0.1, frequency_min_default * 1.1)),
    max=np.log10(frequency_max_limit), step=0.05,
    description="Maximum (Hz):", readout_format=".1f", continuous_update=False,
    style=control_style, layout=widgets.Layout(width="380px"),
)


def plot_cwt(channel, start_s, window_s, preprocessing, wavelet_name, minimum_hz, maximum_hz):
    times, values = read_segment(channel, start_s, window_s, preprocessing)
    try:
        power, frequencies, scales = cwt_power(values, wavelet_name, minimum_hz, maximum_hz)
    except ValueError as error:
        print(f"CWT unavailable: {error}")
        return

    positive_power = power[power > 0]
    if positive_power.size == 0:
        print("CWT unavailable: this segment has zero power at all selected scales.")
        return
    vmin, vmax = np.percentile(positive_power, [1, 99])
    vmin = max(vmin, np.finfo(float).tiny)
    vmax = max(vmax, vmin * 10)

    figure, (trace_axis, cwt_axis) = plt.subplots(
        2, 1, figsize=(15, 8), sharex=True,
        gridspec_kw={"height_ratios": [1, 3]}, constrained_layout=True,
    )
    trace_axis.plot(times, values, color="black", linewidth=0.7)
    trace_axis.set_ylabel("Signal (µV)")
    trace_axis.set_title(
        f"{channel} — CWT ({wavelet_name}) | {times[0]:.3f}–"
        f"{times[-1] + 1/sfreq:.3f} s | {preprocessing}"
    )

    mesh = cwt_axis.pcolormesh(
        times, frequencies, power, shading="auto", cmap="magma",
        norm=LogNorm(vmin=vmin, vmax=vmax), rasterized=True,
    )
    cwt_axis.set_yscale("log")
    cwt_axis.set_ylim(minimum_hz, maximum_hz)
    cwt_axis.set_ylabel("Frequency (Hz, log scale)")
    cwt_axis.set_xlabel("Time (s)")
    cwt_axis.grid(False)
    figure.colorbar(mesh, ax=cwt_axis, label="Wavelet power (µV², log color scale)")
    conservative_edge_s = min(float(np.max(scales)) / sfreq, window_s / 2)
    shade_boundaries(cwt_axis, times, conservative_edge_s)
    cwt_axis.margins(x=0)
    plt.show()


cwt_output = widgets.interactive_output(plot_cwt, {
    "channel": channel_picker, "start_s": start_picker, "window_s": window_picker,
    "preprocessing": preprocessing_picker, "wavelet_name": cwt_wavelet_picker,
    "minimum_hz": cwt_min_picker, "maximum_hz": cwt_max_picker,
})
cwt_tab = widgets.VBox([
    widgets.HBox([cwt_wavelet_picker]),
    widgets.HBox([cwt_min_picker, cwt_max_picker]),
    widgets.HTML(
        f"<small>Frequencies must satisfy 0 &lt; minimum &lt; maximum &lt; "
        f"Nyquist ({nyquist_hz:g} Hz). Shaded ends are padding-sensitive.</small>"
    ),
    cwt_output,
])

## 9. Launch the interactive viewer

Use the shared controls once, then switch between the DWT and CWT tabs. Both views update from the same channel and exact sample interval.


In [10]:
tabs = widgets.Tab(children=[dwt_tab, cwt_tab])
tabs.set_title(0, "DWT components")
tabs.set_title(1, "CWT scalogram")
display(shared_controls, tabs)


## Practical interpretation checklist

1. Confirm the channel montage, units, sampling rate, hardware filters, and reference used during acquisition.
2. Treat coefficients inside the shaded boundary regions cautiously.
3. Compare candidate events with the raw trace and neighboring contacts; wavelets can emphasize artifacts as well as physiology.
4. Avoid comparing absolute CWT color values between views because each view uses robust local color limits.
5. Record the channel, interval, preprocessing choice, wavelet name/parameters, DWT level or CWT frequency limits, and package versions for reproducibility.
6. Do not use this exploratory viewer alone for diagnosis or clinical decisions.
